# Figure 1

Notebook-native smoke reproduction of the phase transition experiment. This version runs the core logic directly inside the notebook instead of shelling out to `run.py`.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys


def in_colab():
    try:
        import google.colab  # type: ignore
        return True
    except ImportError:
        return False


def find_repo_root():
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        if (base / "src" / "ghosts").exists() and (
            (base / "tutorials").exists() or (base / "experiments").exists()
        ):
            return base

    if in_colab():
        repo = Path('/content/ghosts-of-softmax')
        if not repo.exists():
            subprocess.run(
                [
                    'git', 'clone', '--depth', '1',
                    'https://github.com/piyush314/ghosts-of-softmax.git',
                    str(repo),
                ],
                check=True,
            )
        return repo

    raise RuntimeError(
        'Run this notebook from inside the ghosts-of-softmax repository, '
        'or open it in Google Colab so the setup cell can clone the repo automatically.'
    )


REPO = find_repo_root()
SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))


def load_module(name, relative_path):
    path = REPO / relative_path
    module_dir = str(path.parent)
    if module_dir not in sys.path:
        sys.path.insert(0, module_dir)
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


OUTPUT_ROOT = Path('/tmp/ghosts-of-softmax-notebooks')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Repo root: {REPO}")
print(f"Notebook outputs: {OUTPUT_ROOT}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

phase = load_module('fig1_phase_run', 'experiments/phasetransition/run.py')

phase.MAX_STEPS = 120
phase.EVAL_N = 128
phase.RANDOM_DIRS = 8
phase.R_VALUES = np.logspace(-3, np.log10(8.0), 28)

X, y = phase.load_digits_csv()
X_train, X_test, y_train, y_test = phase.stratified_split(X, y, test_ratio=0.30, seed=42)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

eval_rng = np.random.default_rng(1234)
eval_idx = eval_rng.choice(len(X_train_t), size=min(phase.EVAL_N, len(X_train_t)), replace=False)
X_eval_t = X_train_t[eval_idx]
y_eval_t = y_train_t[eval_idx]

seed = 7
model = phase.MLPDigits(width=128)
snapshots = phase.train_snapshots_for_seed(seed, X_train_t, y_train_t, X_test_t, y_test_t)

stage_results = {}
for stage in phase.STAGE_NAMES:
    model.load_state_dict(snapshots[stage]['state'])
    stage_results[stage] = phase.evaluate_stage(
        model,
        X_eval_t,
        y_eval_t,
        seed + 1000 + phase.STAGE_NAMES.index(stage),
    )

summary = []
for stage in phase.STAGE_NAMES:
    out = stage_results[stage]
    summary.append({
        'stage': stage,
        'acc': snapshots[stage]['acc'],
        'onset_grad': phase.first_crossing(phase.R_VALUES, out['grad_curve'], threshold=1.1),
        'onset_rand': phase.first_crossing(phase.R_VALUES, out['rand_median'], threshold=1.1),
        'h_ratio_median': float(np.median(out['h_ratio'])),
    })
summary


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, stage in zip(axes, phase.STAGE_NAMES):
    out = stage_results[stage]
    ax.plot(phase.R_VALUES, out['grad_curve'], color='#E3120B', lw=2.3, label='gradient')
    ax.plot(phase.R_VALUES, out['rand_median'], color='#006BA2', lw=2.3, label='median random')
    ax.fill_between(phase.R_VALUES, out['rand_q25'], out['rand_q75'], color='#006BA2', alpha=0.18)
    ax.axvline(1.0, color='#3D3D3D', ls='--', lw=1.2)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(r'$r = \tau / \rho_a$')
    ax.set_title(f"{stage} (acc={snapshots[stage]['acc']:.2f})")
    ax.grid(True, which='both', alpha=0.25)
axes[0].set_ylabel('Loss ratio')
axes[0].legend(frameon=False)
fig.suptitle('Phase transition from the notebook smoke path', y=1.02)
plt.tight_layout()
plt.show()
